# Riduzione della dimensionalità e clustering

Ispirato a: https://github.com/paolodeangelis/AEM

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

## Dataset - pinguini di Palmer

<img src="https://allisonhorst.github.io/palmerpenguins/reference/figures/lter_penguins.png" width="300"/>

Fonte: https://allisonhorst.github.io/palmerpenguins/articles/intro.html

Il dataset contiene misurazioni anatomiche per tre specie di pinguini, osservate su tre isole nell'[arcipelago di Palmer](https://en.wikipedia.org/wiki/Palmer_Archipelago), in Antartide.

In questo notebook faremo uso di `pandas` per automatizzare la gestione del dataset (pulizia, trasformazioni, analisi, visualizzazione, ...).

In [ ]:
# Scarichiamo il dataset
!wget https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv

La prima fase consiste in una rapida analisi ed esplorazione visiva dei dati. Una metrica utile da considerare è la correlazione tra le variabili numeriche.

In [ ]:
plt.style.use("default")

# Importiamo con pandas il dataset usando il metodo read_csv()
# (csv = comma-separated values)
penguins_data = pd.read_csv("penguins.csv")

# Puliamo i dati (non vogliamo NaNs)
penguins_data = penguins_data.dropna(axis=0, how='any')

# Rimuoviamo gli anni (la progressione temporale non ci interessa)
penguins_data = penguins_data.drop('year', axis=1)

# Ulteriore operazioni di pulizia: 
# definiamo esplicitamente i dati categorici come categorie
for col in ['species','island','sex']:
    penguins_data[col] = penguins_data[col].astype('category')

penguins_data

Vocabolario anatomico: **bill** = becco, **flipper** = pinna.

Ricordate che PCA è fondamentalmente una decomposizione ai valori singolari della matrice di convarianza delle features. Esaminiamo dunque la correlazione tra features, descritta (ad esempio) dal [coefficiente di correlazione di Pearson](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient):

$$ \rho_{X,Y}=\frac{\text{cov}(X,Y)}{\sigma_X\sigma_Y} $$

Dove $X$ e $Y$ sono le variabili aleatorie associate ad una coppia di features. 

Breve recap su varianza e covarianza:

$$\sigma^2_X = \mathbb{E}[X^2]-\mathbb{E}[X]^2 \; , \quad \text{cov}(X,Y)=\mathbb{E}[XY]-\mathbb{E}[X]\mathbb{E}[Y]$$

Notare che $\rho_{X,X}=1$, poiché $\text{cov}(X,X)=\sigma^2_X$.

In [ ]:
help(penguins_data.corr)

In [ ]:
penguins_corr = penguins_data.corr(numeric_only=True)
penguins_corr.style.background_gradient(cmap="coolwarm")

Valori positivi (rossi) = features correlate.

Valori negativi (blu) = features anti-correlate.

Fissiamo una categoria e visualiziamo una coppia e una tripletta di features:

In [ ]:
%matplotlib ipympl
fig1 = plt.figure(figsize=(9, 6))

# Try to change the fields
data_axis = ["body_mass_g","bill_length_mm"]

# Try to change the category
category_label = 'species'
# category_label = 'island'
# category_label = 'sex'

ax0 = fig1.add_subplot(111)
for label in penguins_data[category_label].unique():
    # plot each specie with a different colour
    sub_set = penguins_data.loc[
        penguins_data[category_label] == label
    ]
    ax0.scatter(sub_set[data_axis[0]], sub_set[data_axis[1]], label=label)
ax0.legend()
ax0.set_xlabel(data_axis[0])
ax0.set_ylabel(data_axis[1])
plt.show()

In [ ]:
fig_3d = plt.figure()
ax_3d = fig_3d.add_subplot(projection='3d')

data_axis_3d = ["body_mass_g","bill_length_mm","flipper_length_mm"]

for label in penguins_data[category_label].unique():
    sub_set = penguins_data.loc[
        penguins_data[category_label] == label
    ]
    ax_3d.scatter(sub_set[data_axis_3d[0]], 
                  sub_set[data_axis_3d[1]], 
                  sub_set[data_axis_3d[2]], label=label)
ax_3d.legend()
ax_3d.set_xlabel(data_axis[0])
ax_3d.set_ylabel(data_axis[1])
plt.show()

## Normalizzare il dataset

Ci sono varie metodologie per normalizzare un dataset, noi utilezzeremo la _standardizzazione Gaussiana_. Per ogni feature, mappiamo:
    
$$ \tilde{X} = \dfrac{X-\mu}{\sigma } $$

Dove $\mu$ e $\sigma$ sono rispettivamente la media e la deviazione standard (per ogni feature).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Tutti i label associati a features numeriche
X_raw = penguins_data.iloc[:, 2:6].values.astype(float)

# Selezioniamo la categoria "species" (provate a ripete con le altre due!)
# 0 : species
# 1 : island
# -1 : sex
index_category = 0
Y = penguins_data.iloc[:, index_category].values.astype(str)

normalizer = StandardScaler()
X = normalizer.fit_transform(X_raw)

print("mean: ", X.mean(axis=0))
print("std: ", X.std(axis=0))

Se tutto e andato bene, la media dovrebbe essere = 0, e la deviazione standard = 1 ("up to machine epsilon").

Gli arrays `X` e `Y` ora contengono rispettivamente:
- Le features agglomerate e normalizzate ($N\times4$);
- Le etichette relative al campo `"species"` ($N\times1$).

In [ ]:
print(X)
print("X.shape =",X.shape)
print(Y)
print("Y.shape =",Y.shape)

## Principal Component Analysis (PCA)

Breve recap sugli aspetti fondamentali di PCA:
1. Essendo un metodo di _unsupervised learning_, abbiamo bisogno soltanto di `X` (i.e. per ora ignoriamo `Y`);
2. PCA opera una diagonalizzazione (o una decomposizione ai valori singolari) della matrice di covarianza delle features.

Invece definire la matrice di covarianza e il problema agli autovalor/valori songolari _a mano_, useremo l'oggetto `PCA` in `sklearn.decomposition`:

In [ ]:
from sklearn.decomposition import PCA

# Numero di componenti (max. 4)
D_PCA = 3

pca = PCA(n_components=D_PCA)
X_pca = pca.fit_transform(X)

Fatto! Take-home message: **do not reinvent the weel!** Se esistono librerie che fanno esattamente ciò che vi serve, usatele. Nel caso della regressione polinomiale, la descrizione matematica aveva puri fini pedagogici.

Ora visualizziamo le componenti principali, colorate per categoria, contro le features originali del dataset:

In [ ]:
fig2 = plt.figure(figsize=(12, 6))

data_axis = ["body_mass_g","bill_length_mm"]

ax1 = fig2.add_subplot(121)
ax2 = fig2.add_subplot(122)

# Visualizziamo il dataset originale insieme alle prime 2 componenti principali
for label in penguins_data["species"].unique():
    indexs = np.where(Y == label)[0]
    sub_set = penguins_data.loc[
        penguins_data["species"] == label
    ]
    ax1.scatter(sub_set[data_axis[0]], sub_set[data_axis[1]], label=label)
    ax2.scatter(X_pca[indexs, 0], X_pca[indexs, 1], label=label)
ax1.legend()
ax1.set_xlabel(data_axis[0])
ax1.set_ylabel(data_axis[1])
ax2.set_xlabel("1st component")
ax2.set_ylabel("2nd component")
plt.show()

In [ ]:
fig_3d_pca = plt.figure()
ax_3d_pca = fig_3d_pca.add_subplot(projection='3d')

data_axis_3d = ["body_mass_g","bill_length_mm","flipper_length_mm"]

# Visualizziamo le prime 3 componenti principali
for label in penguins_data["species"].unique():
    indexs = np.where(Y == label)[0]
    sub_set = penguins_data.loc[
        penguins_data["species"] == label
    ]
    ax_3d_pca.scatter(X_pca[indexs, 0], 
                      X_pca[indexs, 1], 
                      X_pca[indexs, 2], label=label)
ax_3d_pca.legend()
ax_3d_pca.set_xlabel("1st component")
ax_3d_pca.set_ylabel("2nd component")
ax_3d_pca.set_zlabel("3rd component")
plt.show()

# Unsupervised clustering

Idea: vogliamo suddividere il dataset processato dalla PCA in "gruppetti", ad esempio per ricostruire le categorie originali.

**NB!** Questo non è un metodo di _classificazione_, non confondetevi! Fino ad ora (spoiler: fino all’ultima cella del notebook) non abbiamo usato i labels per allenare il modello, ma semplicemente per visualizzare le feature e le componenti principali. L’informazione in `Y` non è stata utilizzata (infatti si tratta di metodi di _unsupervised_ learning).

## Gaussian Mixtures

![EM Clustering](https://upload.wikimedia.org/wikipedia/commons/6/69/EM_Clustering_of_Old_Faithful_data.gif)

Assumiamo che i dati in ogni gruppetto (cluster) siano disposti con una probabiltà Gaussiana multidimensionale. L'algoritmo di Gaussian Mixtures assume che ci siano `n_components` = *numero di cluster* Gaussiane, e i parametri delle gaussiane ($\mu$, $\sigma$) vengono trovati per massimizzare il [Maximum Likelihood Estimate (MLE)](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation).

In [ ]:
from sklearn.mixture import GaussianMixture

# Numero di 'mixtures', i.e. clusters
D_GM = 3

cluster_model = GaussianMixture(
    n_components=D_GM,
    covariance_type="full",  # Stima/calcolo della matrice di covarianza
)

# Training (NB non usiamo Y)
cluster_model.fit(X_pca)
Y_prediction = cluster_model.predict(X_pca)

Valutiamo il risultato del clustering, prima in modo qualitativo plottando le categorie originali vs. i clusters predetti dal modello:

In [ ]:
fig3 = plt.figure(figsize=(12, 6))

ax1 = fig3.add_subplot(121)
ax2 = fig3.add_subplot(122)
for label in penguins_data["species"].unique():
    indexs = np.where(Y == label)[0]
    sub_set = penguins_data.loc[
        penguins_data["species"] == label
    ]  # plotto ogni specie con colori diversi
    ax1.scatter(X_pca[indexs, 0], X_pca[indexs, 1], label=label)
for clust_id in np.unique(Y_prediction):
    indexs = np.where(Y_prediction == clust_id)[0]
    ax2.scatter(X_pca[indexs, 0], X_pca[indexs, 1], label=f"cluster {clust_id}")
for ax in [ax1, ax2]:
    ax.legend()
    ax.set_xlabel("1st component")
    ax.set_ylabel("2nd component")
plt.show()

Valutiamo ore il modello in modo quantitativo.

Per fare ciò dobbiamo usare una metrica che misura la bontà del nostro modello. Considerando che la numerazione dei cluster non è importante, dobbiamo usare una _metrica_ (distanza matematica) indipendente dalla permutazione dei labels in `Y`, `Y_prediction`.

In questo esercizio usiamo una metrica abbastaza comune, ovvero la **Mutual Info Score** (IT: [Informazione mutua](https://it.wikipedia.org/wiki/Informazione_mutua))

$
\text{MI}(U,V) = 
\sum_{i=1}^{|U|} \sum_{j=1}^{|V|} \frac{|U_i\cap V_j|}{N} \log\frac{N|U_i \cap V_j|}{|U_i||V_j|}
$

Osservate come la misura della infomazione $I = |A|\log(|A|) = p_{A}\log(p_{A})$ assomigli molto alla definzione di Gibbs dell'**entropia** usata in Maccanica Statistica $S = k_{B} \sum_i p_{i}\log(p_{i})$. Non è una coincidenza!

In [ ]:
from sklearn.metrics import mutual_info_score

mi_score = mutual_info_score(Y, Y_prediction)
print(f"Mutual Information Score = {mi_score*100:1.2f} %")

### Interpretazione

**MI score alta**: i clusters "etichettano" in maniera equivalente alle categorie originali. Cluster e categorie contengono la stessa informazione, possono essere la "proxi" gli uni delle altre. "Just fit".

**MI score bassa**: i clusters "etichettano" in maniera diversa dalle categorie. L'informazione codificata non è equivalente. Collegandoci al discorso di "bias/variance tradeoff", può essere dovuto ad una mancanza di precisione e/o accuratezza del metodo. Ad esempio, le categorie possono essere effettivamente sepaarte in clusters, ma il nostro modello non è abbastanza buono per separarle. Oppure le categorie originali contengono la stessa informazione (e.g. l'anatomia non dipende dalla specie), e ogni algoritmo con più di un cluster andrà in overfitting.